In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.io import loadmat

from cqpsolver import Problem, Solver, SolverState

In [2]:
mat_dict: dict[np.ndarray] = loadmat("../QP-Test-Problems/MAT_Files/EXDATA.mat")

Q: sp.csc_array = sp.csc_array(mat_dict["Q"].astype(float).toarray())
q: sp.csc_array = sp.csc_array(mat_dict["c"].astype(float))
A: np.ndarray = mat_dict["A"].astype(float).toarray()
rl: np.ndarray = mat_dict["rl"].astype(float).flatten()
ru: np.ndarray = mat_dict["ru"].astype(float).flatten()
lb: np.ndarray = mat_dict["lb"].astype(float).flatten().reshape(-1, 1)
ub: np.ndarray = mat_dict["ub"].astype(float).flatten().reshape(-1, 1)

In [3]:
eq_mask: np.ndarray = rl == ru
A_eq: np.ndarray = A[eq_mask]
b_eq: np.ndarray = ru[eq_mask].reshape(-1, 1)

A_eq: np.ndarray = A_eq if A_eq.size > 0 else np.zeros((0, A.shape[1]))
A_eq: sp.csc_array = sp.csc_array(A_eq)
b_eq: np.ndarray = b_eq if b_eq.size > 0 else np.zeros((0, 1))
b_eq: sp.csc_array = sp.csc_array(b_eq)

ineq_mask: np.ndarray = np.invert(eq_mask)
G_ineq: np.ndarray = np.vstack([A[ineq_mask], -A[ineq_mask]])
h_ineq: np.ndarray = np.concatenate([ru[ineq_mask], -rl[ineq_mask]]).reshape(-1, 1)

G_full: np.ndarray = np.vstack([G_ineq, np.eye(Q.shape[0]), -np.eye(Q.shape[0])])
h_full: np.ndarray = np.vstack([h_ineq, ub, -lb])

G: sp.csc_array = sp.csc_array(G_full[np.isfinite(h_full).flatten()])
h: sp.csc_array = sp.csc_array(h_full[np.isfinite(h_full).flatten()]).reshape(-1, 1)

In [4]:
prob: Problem = Problem(Q, q, G, h, A_eq, b_eq)
solver: Solver = Solver(prob, max_iter=100)
state_history: list[SolverState] = solver.solve()
final = state_history[-1]
print(f"Iter: {final.iter}, Converged: {bool(final.residuals.converged(tol=1e-8))}")
print(f"Obj: {final.obj}")
print(f"Primal ineq: {final.residuals.primal_ineq}")
print(f"Primal eq: {final.residuals.primal_eq}")
print(f"Stationarity: {final.residuals.stationarity}")
print(f"Duality: {final.residuals.duality}")

Iter: 18, Converged: True
Obj: -141.84343218893025
Primal ineq: 5.622554988882454e-14
Primal eq: 1.0552171369206694e-13
Stationarity: 1.7004206310391502e-12
Duality: 4.519322145666632e-10
